# 01 Data cleaning
One row per England LSOA. Each step prints row counts so the cleaning is auditable.
Run from the `notebooks/` folder.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
from scipy.spatial import cKDTree
RAW = Path("../data/raw"); OUT = Path("../data/clean"); OUT.mkdir(parents=True, exist_ok=True)
log = []  # cleaning log: (step, rows_before, rows_after)

## 1. LSOA boundaries (2021, BSC)
Keep England only (codes start with E), project to British National Grid (metres).

In [ ]:
lsoa = gpd.read_file(RAW / "lsoa_boundaries_2021_bsc.geojson")
n0 = len(lsoa)
lsoa = lsoa[lsoa["LSOA21CD"].str.startswith("E")].to_crs(27700)
lsoa = lsoa[["LSOA21CD","LSOA21NM","RUC21CD","Urban_rura","geometry"]].copy()
lsoa["area_km2"] = lsoa.geometry.area / 1e6
lsoa["cent_x"] = lsoa.geometry.centroid.x   # geometric centroid (limitation)
lsoa["cent_y"] = lsoa.geometry.centroid.y
log.append(("Boundaries: keep England", n0, len(lsoa))); log[-1]

## 2. Population (mid-2024)

In [ ]:
pop = pd.read_excel(RAW/"lsoa_population_2022_2024.xlsx", sheet_name="Mid-2024 LSOA 2021",
                    header=3, usecols=["LSOA 2021 Code","Total"])
pop.columns = ["LSOA21CD","population"]
pop.describe()

## 3. IMD 2025

In [ ]:
imd = pd.read_csv(RAW/"imd_2025.csv").rename(columns={
    "LSOA code (2021)":"LSOA21CD",
    "Index of Multiple Deprivation (IMD) Score":"imd_score",
    "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)":"imd_decile"
})[["LSOA21CD","imd_score","imd_decile"]]
imd.describe()

## 4. NaPTAN bus stops
Keep active stops of type BCT/BCS (on-street bus stops), drop rows without coordinates.

In [ ]:
stops = pd.read_csv(RAW/"naptan.csv", low_memory=False,
                    usecols=["ATCOCode","StopType","Status","Easting","Northing"])
n0 = len(stops)
stops = stops[(stops.Status=="active") & stops.StopType.isin(["BCT","BCS"])]
log.append(("NaPTAN: active BCT/BCS", n0, len(stops)))
n1 = len(stops)
stops = stops.dropna(subset=["Easting","Northing"])
log.append(("NaPTAN: drop missing coordinates", n1, len(stops))); log[-2:]

## 5. Stops per LSOA and distance to nearest stop

In [ ]:
sg = gpd.GeoDataFrame(stops, geometry=gpd.points_from_xy(stops.Easting, stops.Northing), crs=27700)
j = gpd.sjoin(sg, lsoa[["LSOA21CD","geometry"]], how="inner", predicate="within")
counts = j.groupby("LSOA21CD").size().rename("n_stops")
log.append(("Stops matched to an LSOA", len(stops), len(j)))
tree = cKDTree(np.c_[stops.Easting, stops.Northing])
lsoa["dist_nearest_stop_m"], _ = tree.query(np.c_[lsoa["cent_x"], lsoa["cent_y"]])
log[-1]

## 6. Join and checks

In [ ]:
df = (lsoa.drop(columns="geometry").merge(pop,on="LSOA21CD",how="left")
      .merge(imd,on="LSOA21CD",how="left").merge(counts,on="LSOA21CD",how="left"))
df["n_stops"] = df["n_stops"].fillna(0).astype(int)
df["pop_density"] = df["population"] / df["area_km2"]
assert df["LSOA21CD"].is_unique
print("Missing values per column:"); print(df.isna().sum())
df.describe().T

## 7. Save and cleaning log

In [ ]:
df.to_parquet(OUT/"lsoa_base.parquet", index=False)
cleaning_log = pd.DataFrame(log, columns=["Step","Rows before","Rows after"])
cleaning_log.to_csv(OUT/"cleaning_log.csv", index=False)
cleaning_log

## 8. Quick look (for the report)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,2, figsize=(10,3.5))
df["dist_nearest_stop_m"].clip(upper=2000).hist(bins=60, ax=ax[0]); ax[0].set_title("Distance to nearest stop (m, capped 2 km)")
df["n_stops"].clip(upper=60).hist(bins=60, ax=ax[1]); ax[1].set_title("Bus stops per LSOA (capped 60)")
plt.tight_layout()